# Citation graph with OpenAlex

This notebook builds a directed graph for the same selected PDFs as the clustering notebooks. **An edge A → B means A cites B.** It uses DOI batches to retrieve work identities and reference lists together, avoiding one reference request per paper.

The complete Semantic Scholar workflow is preserved in `Citation_graph.ipynb`. This notebook is the faster alternative; do not run both retrievals simultaneously. All stages are shown below and implemented in `src/batill/openalex.py`. PDFs and embeddings are reused.

## Method and limits

1. Load the current PDF selection and exact-title bibliography metadata. Reuse approved Semantic Scholar DOI metadata when available, without making Semantic Scholar requests.
2. Resolve DOIs in batches of up to 100. Prefer the bibliography DOI over an approved alternative version. Apply the recorded PDF-title-page review in `configs/openalex_doi_overrides.json`.
3. For gaps, search titles conservatively. Automatic search acceptance requires an exact normalized title of at least four words, an exact known year, and no similarly scored competing title. Ordinary DOI matches with title similarity below 0.8 remain review items. This score is a screening heuristic, not a confidence probability.
4. Build edges between selected, approved identities. Keep all selected PDFs as nodes, including unresolved and isolated ones. Hold duplicate identities rather than count a work twice.

Each source uses **one** OpenAlex record's reference list. A verified alternative DOI can identify a cited target, but its references are not pooled with the source list. Working-paper versions are allowed under the project's review policy; their bibliographies can differ from the local PDFs. OpenAlex reference lists are not guaranteed complete. An empty list or an isolated node is **not** evidence of an irrelevant paper.

The raw API snapshot, matching decisions, graph tables and sparse matrix are saved. Later citation clustering can use direct links or **bibliographic coupling** (shared references), including references outside the selected corpus.

API documentation: [batch DOI requests and reference lists](https://help.openalex.org/how-to/api-recipes/) and [authentication](https://help.openalex.org/api/authentication/). A limited keyless budget is available; a key can be supplied privately if necessary.

**Update after manual review:** Sections 1–6 preserve the original API baseline. Section 7 applies the recorded manual corrections and exports the graph to use next under `reviewed/`: one node per unique work, a mapping from every PDF, and explicitly partial PDF-derived reference additions.

## 1. Load the selected corpus

Use the `ml` kernel with this project installed. If dependencies are missing, run `python -m pip install -e '.[citations,notebook]'` in a terminal from the project folder.

Paper IDs and PDF hashes stay aligned with the embeddings. The current exclusion file determines membership. Saved Semantic Scholar review metadata are validated before reuse; the fallback for a fresh project is the local bibliography alone.

In [1]:
from pathlib import Path
import sys

# Find the source package before importing project modules.
ROOT = next((p for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             for p in (parent, parent / 'batill')
             if (p / 'src' / 'batill').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the batill project folder.')
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
print('Python:', sys.executable)

import os
import time
import pandas as pd
from IPython.display import display, Markdown
from batill.openalex import OpenAlexClient, load_inputs, resolve_works, export_graph
from batill.storage import read_json, write_json, file_hash


papers, exclusion_audit, selection = load_inputs(ROOT)
RUN_DIR = ROOT / 'outputs/citation_graph_openalex' / selection['selection_id'][:16]
RUN_DIR.mkdir(parents=True, exist_ok=True)
papers.to_csv(RUN_DIR / 'selected_papers.csv', index=False)
exclusion_audit.to_csv(RUN_DIR / 'exclusion_audit.csv', index=False)
display(pd.Series({
    'selected PDFs': len(papers),
    'local bibliography DOI': int(papers.doi.ne('').sum()),
    'local or previously approved DOI': int((papers.doi.ne('') | papers.approved_version_doi.ne('')).sum()),
}).to_frame('papers'))
print('Results:', RUN_DIR)

,papers
selected PDFs,294
local bibliography DOI,230
local or previously approved DOI,283


Results: /Users/fdedek/projects/batill/outputs/citation_graph_openalex/9ad47e071ee12c2f


## 2. API access and documented identity corrections

Start without a key. If OpenAlex returns an authentication or quota error, set `PROMPT_FOR_KEY = True` and enter an **OpenAlex** key at the hidden prompt (the Semantic Scholar key is for a different service). Alternatively, set `OPENALEX_API_KEY` in the kernel environment. The key is not written to output files.

Successful responses are cached. Rerunning uses the same snapshot and only requests missing queries. Errors stop promptly instead of entering a long retry loop. The recorded DOI overrides below are pinned to immutable PDF hashes; their reasons and PDF evidence remain available for review.

In [2]:
from getpass import getpass

PROMPT_FOR_KEY = False
api_key = os.environ.get('OPENALEX_API_KEY')
if PROMPT_FOR_KEY:
    api_key = getpass('OpenAlex API key: ').strip()
client = OpenAlexClient(ROOT / 'outputs/citation_graph_openalex/api_cache', api_key=api_key)
del api_key

review_path = ROOT / 'configs/openalex_doi_overrides.json'
review_records = read_json(review_path)['overrides'] if review_path.exists() else {}
review_records = {key: record for key, record in review_records.items() if key in set(papers.Key)}
DOI_OVERRIDES = {key: record['doi'] for key, record in review_records.items()}
review_table = pd.DataFrame([{'Key': key, **record} for key, record in review_records.items()])
review_table.to_csv(RUN_DIR / 'review_decisions.csv', index=False)
display(review_table[['paper_id', 'doi', 'version_status', 'reason']] if len(review_table) else review_table)

,paper_id,doi,version_status,reason
0,P028,10.1016/0304-405x(89)90080-9,publication_identity_confirmed,PDF title and Baker/Wruck authors agree; OpenA...
1,P073,10.1093/rfs/hhaf111,publication_identity_confirmed,DOI printed on the PDF first page.
2,P084,10.1016/s0304-405x(98)00013-0,publication_identity_confirmed,Exact bibliographic DOI; title and all three a...
3,P127,10.2139/ssrn.4641071,same_work_version_not_verified,Title and all three authors agree with PDF fir...
4,P154,10.1111/fima.12264,publication_identity_confirmed,DOI on PDF first page; same work as P155. Keep...
5,P155,10.1111/fima.12264,publication_identity_confirmed,DOI on PDF first page; same work as P154. Keep...
6,P159,10.1093/ser/mwv026,publication_identity_confirmed,Bibliographic DOI and review-symposium title a...
7,P170,10.67929/ecgi-law-086-2007,same_work_version_not_verified,Exact title and Gilson/Whitehead authors agree...
8,P189,10.2139/ssrn.3535677,same_work_version_not_verified,Exact title and Gredil/Liu/Sensoy authors agre...
9,P220,10.2139/ssrn.4271808,same_work_version_not_verified,Exact title and Jiang/Mason/Qian/Utke authors ...


## 3. Retrieve work records and review coverage

This cell performs the batch DOI requests and a small number of searches for remaining gaps. Each response already contains `referenced_works`; no additional call is needed per source bibliography.

The matching table distinguishes accepted identities, unmatched/search proposals and duplicate identities. Duplicate OpenAlex entries for one DOI are also held unless another safe lookup resolves the work. No PDFs are deleted or excluded from the main analysis by this notebook.

Read `matches.csv`, `search_candidates.csv`, `ambiguous_provider_dois.json`, the override configuration and `review/` evidence before changing decisions. The elapsed time below includes cached responses on repeat runs; it is not a fresh-download benchmark.

In [3]:
start = time.monotonic()
matches, works, aliases = resolve_works(
    client, papers, RUN_DIR, overrides=DOI_OVERRIDES, search_missing=True,
)
print(f'Resolution finished in {time.monotonic() - start:.1f}s; new HTTP requests: {client.network_requests}')
display(matches.groupby(['match_status', 'approved']).size().rename('papers').to_frame())
pending = matches.loc[~matches.approved, [
    'paper_id', 'query_title', 'openalex_title', 'openalex_id', 'match_status',
]]
pending.to_csv(RUN_DIR / 'pending_review.csv', index=False)
display(pending)

DOI batch: 100/345 identifiers


DOI batch: 200/345 identifiers


DOI batch: 300/345 identifiers


DOI batch: 345/345 identifiers


Resolution finished in 0.1s; new HTTP requests: 0


,,papers
match_status,approved,
accepted_doi,True,288
accepted_exact_title_year,True,1
duplicate_work_identity_review_required,False,4
search_needs_review,False,1


,paper_id,query_title,openalex_title,openalex_id,match_status
19,P020,Why are buyouts levered? the financial structu...,Why Are Buyouts Levered? The Financial Structu...,https://openalex.org/W3122362577,duplicate_work_identity_review_required
146,P154,Ljungqvist - The investment behavior of buyout...,The investment behavior of buyout funds: Theor...,https://openalex.org/W2913825478,duplicate_work_identity_review_required
147,P155,Ljungqvist - The investment behavior of buyout...,The investment behavior of buyout funds: Theor...,https://openalex.org/W2913825478,duplicate_work_identity_review_required
277,P301,Valuing private equity,Valuing Private Equity,https://openalex.org/W2891860082,search_needs_review
284,P308,Why are buyouts levered? the financial structu...,Why Are Buyouts Levered? The Financial Structu...,https://openalex.org/W3122362577,duplicate_work_identity_review_required


## 4. Export the graph and assess missing data

`nodes.csv` contains every selected PDF. `edges.csv` contains unique directed internal citations. `all_references.csv` also retains references outside the corpus for later bibliographic coupling. References to verified alternative DOI versions are labelled in the edge table.

`adjacency.npz` is a sparse matrix: **row cites column**. Its order is explicitly saved in `adjacency_order.csv`; use PDF hashes when joining to embeddings. `citation_graph.graphml` is an equivalent graph export.

Interpret connected components and isolates alongside identity and reference coverage. A source with an empty API reference list may still receive incoming citations. The raw cache preserves the retrieval snapshot; refreshing it later can change the results.

In [4]:
nodes, edges, graph, summary = export_graph(
    matches, works, aliases, RUN_DIR, selection, client,
)
# Record the human-readable review configuration alongside machine matching settings.
manifest = read_json(RUN_DIR / 'run_manifest.json')
manifest['review_config_sha256'] = file_hash(review_path) if review_path.exists() else None
write_json(RUN_DIR / 'run_manifest.json', manifest)
display(pd.Series({k: v for k, v in summary.items() if k != 'weak_component_sizes'}).to_frame('value'))
print('Weak component sizes:', summary['weak_component_sizes'])
display(nodes.loc[nodes.approved & ~nodes.reference_status.eq('available_nonempty'),
                  ['paper_id', 'query_title', 'reference_status']])
print('Outputs saved in:', RUN_DIR)

,value
provider,OpenAlex
selected_pdfs,294
approved_work_identities,289
unapproved_pdfs,5
approved_nonempty_reference_lists,281
approved_empty_reference_lists,8
approved_missing_reference_lists,0
internal_citation_edges,2396
all_reference_links,18024
edges_using_version_alias,116


Weak component sizes: [279, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


,paper_id,query_title,reference_status
31,P032,An Empirical Guide to Investor-Level Private E...,empty_provider_list
62,P064,What Do Different Commercial Data Sets Tell Us...,empty_provider_list
65,P067,Private equity portfolio companies A first loo...,empty_provider_list
71,P073,The Structure of Leveraged Buyouts and the Fre...,empty_provider_list
161,P170,"Deconstructing equity: Public ownership, agenc...",empty_provider_list
173,P187,Evaluating Private Equity performance using St...,empty_provider_list
219,P236,Private equity for pension plans? Evaluating p...,empty_provider_list
244,P263,Does private equity create wealth? the effects...,empty_provider_list


Outputs saved in: /Users/fdedek/projects/batill/outputs/citation_graph_openalex/9ad47e071ee12c2f


## 5. Inspect individual links

The ranking below counts **citations from other selected papers**, not global citation counts. Use the second table to inspect what one paper cites and what cites it. These checks help assess whether the graph looks sensible before choosing a citation-clustering method.

For later analysis, compare citation-based communities with the existing text-based clusters. Shared-reference clustering is also possible using `all_references.csv`; these are two different relations and should be labelled accordingly.

In [5]:
display(nodes.nlargest(15, 'internal_in_degree')[
    ['paper_id', 'query_title', 'internal_in_degree', 'internal_out_degree']
])
PAPER_ID = 'P003'
selected = nodes.loc[nodes.paper_id.eq(PAPER_ID)]
if selected.empty:
    raise ValueError('Choose a paper_id from nodes.csv')
titles = nodes.set_index('paper_id').query_title
links = edges.loc[edges.source_paper_id.eq(PAPER_ID) | edges.target_paper_id.eq(PAPER_ID)].copy()
links['source_title'] = links.source_paper_id.map(titles)
links['target_title'] = links.target_paper_id.map(titles)
display(links[['source_paper_id', 'source_title', 'target_paper_id', 'target_title', 'target_match_method']])

,paper_id,query_title,internal_in_degree,internal_out_degree
211,P227,"Private equity performance: Returns, persisten...",111,3
213,P229,Leveraged buyouts and private equity,91,29
209,P224,The Effects of Management Buyouts on Operating...,87,2
177,P191,Do Buyouts (Still) Create Value?,62,13
247,P266,The economics of private equity funds,61,7
185,P199,Private equity performance: What do we know?,57,5
229,P247,"Smart institutions, foolish choices: The limit...",51,3
20,P021,"Borrow Cheap, Buy High? The Determinants of Le...",46,16
264,P287,The performance of private equity funds,43,5
2,P003,Corporate governance and value creation: Evide...,41,16


,source_paper_id,source_title,target_paper_id,target_title,target_match_method
16,P003,Corporate governance and value creation: Evide...,P300,Corporate Ownership Structure and Performance ...,primary_work_id
17,P003,Corporate governance and value creation: Evide...,P281,Consequences of leveraged buyouts,primary_work_id
18,P003,Corporate governance and value creation: Evide...,P078,The performance of reverse leveraged buyouts,verified_doi_version_alias
19,P003,Corporate governance and value creation: Evide...,P266,The economics of private equity funds,primary_work_id
20,P003,Corporate governance and value creation: Evide...,P110,"Private equity, leveraged buyouts and governance",primary_work_id
21,P003,Corporate governance and value creation: Evide...,P224,The Effects of Management Buyouts on Operating...,primary_work_id
22,P003,Corporate governance and value creation: Evide...,P228,The Evolution of Buyout Pricing and Financial ...,primary_work_id
23,P003,Corporate governance and value creation: Evide...,P227,"Private equity performance: Returns, persisten...",verified_doi_version_alias
24,P003,Corporate governance and value creation: Evide...,P290,Why do public firms go private in the UK? The ...,primary_work_id
26,P003,Corporate governance and value creation: Evide...,P191,Do Buyouts (Still) Create Value?,primary_work_id


## 6. Compact saved report

This report describes the current snapshot rather than claiming complete PDF bibliography recovery. It is regenerated from the graph tables so its counts stay consistent when the selection changes.

In [6]:
pending_lines = '\n'.join(
    f"- {r.paper_id}: {r.query_title} — {r.match_status}"
    for r in nodes.loc[~nodes.approved].itertuples()
)
report = f"""# OpenAlex citation graph: retrieval snapshot

- Selected PDFs: **{summary['selected_pdfs']}**.
- Approved unique work identities: **{summary['approved_work_identities']}**.
- Approved sources with nonempty reference lists: **{summary['approved_nonempty_reference_lists']}**.
- Approved sources with empty lists: **{summary['approved_empty_reference_lists']}**; completeness is unknown.
- Internal directed citation edges: **{summary['internal_citation_edges']}**.
- Reference links retained, including outside works: **{summary['all_reference_links']}**.
- Largest weakly connected component: **{max(summary['weak_component_sizes'], default=0)} PDFs**.
- Isolated PDFs: **{summary['isolated_pdfs']}**, including unresolved identities.
- Edges matched through alternative DOI versions: **{summary['edges_using_version_alias']}**.

A → B means A cites B. All selected PDFs remain in nodes.csv. Duplicate identities are held rather than double-counted. A missing edge is not proof that no citation exists: provider coverage and paper versions can differ from the PDFs. Working-paper versions are permitted by the project's review policy.

## Unresolved identities

{pending_lines}

## Method and evidence

The complete workflow is in Citation_graph_OpenAlex.ipynb and src/batill/openalex.py. DOI metadata come from local bibliography entries or approved earlier identity matches. Additional DOI decisions are pinned to PDF hashes in configs/openalex_doi_overrides.json, with first-page evidence in review/pdf_identity_evidence.json. Automatic title matching is conservative; other candidates remain review items. The earlier Semantic Scholar notebook and cached responses are preserved.

Each source contributes one reference list. Alternative DOI versions may identify a target but do not contribute additional source references. The directed internal graph is in edges.csv and citation_graph.graphml; all_references.csv supports later shared-reference analysis. Use adjacency_order.csv with adjacency.npz. Matching settings, input fingerprint, code hash and raw-cache checksums are recorded in run_manifest.json. The API cache supports offline reproduction of this snapshot.

Before comparing clusters, inspect unresolved identities and missing reference coverage. Bibliographic coupling can supplement direct citation clustering; neither method guarantees one unique correct number of clusters.
"""
(RUN_DIR / 'report.md').write_text(report, encoding='utf-8')
display(Markdown(report))

# OpenAlex citation graph: retrieval snapshot

- Selected PDFs: **294**.
- Approved unique work identities: **289**.
- Approved sources with nonempty reference lists: **281**.
- Approved sources with empty lists: **8**; completeness is unknown.
- Internal directed citation edges: **2396**.
- Reference links retained, including outside works: **18024**.
- Largest weakly connected component: **279 PDFs**.
- Isolated PDFs: **12**, including unresolved identities.
- Edges matched through alternative DOI versions: **116**.

A → B means A cites B. All selected PDFs remain in nodes.csv. Duplicate identities are held rather than double-counted. A missing edge is not proof that no citation exists: provider coverage and paper versions can differ from the PDFs. Working-paper versions are permitted by the project's review policy.

## Unresolved identities

- P020: Why are buyouts levered? the financial structure of private equity funds — duplicate_work_identity_review_required
- P154: Ljungqvist - The investment behavior of buyout funds  Theory and evidence (1) 2 — duplicate_work_identity_review_required
- P155: Ljungqvist - The investment behavior of buyout funds  Theory and evidence — duplicate_work_identity_review_required
- P301: Valuing private equity — search_needs_review
- P308: Why are buyouts levered? the financial structure of private equity funds — duplicate_work_identity_review_required

## Method and evidence

The complete workflow is in Citation_graph_OpenAlex.ipynb and src/batill/openalex.py. DOI metadata come from local bibliography entries or approved earlier identity matches. Additional DOI decisions are pinned to PDF hashes in configs/openalex_doi_overrides.json, with first-page evidence in review/pdf_identity_evidence.json. Automatic title matching is conservative; other candidates remain review items. The earlier Semantic Scholar notebook and cached responses are preserved.

Each source contributes one reference list. Alternative DOI versions may identify a target but do not contribute additional source references. The directed internal graph is in edges.csv and citation_graph.graphml; all_references.csv supports later shared-reference analysis. Use adjacency_order.csv with adjacency.npz. Matching settings, input fingerprint, code hash and raw-cache checksums are recorded in run_manifest.json. The API cache supports offline reproduction of this snapshot.

Before comparing clusters, inspect unresolved identities and missing reference coverage. Bibliographic coupling can supplement direct citation clustering; neither method guarantees one unique correct number of clusters.


## 7. Apply manual identity and citation review

This is the **final graph for the next clustering step**. The API baseline above remains unchanged for comparison.

- Publisher metadata confirm P020/P308 and P154/P155 as duplicate PDF pairs. Use P020 and P154 as canonical graph nodes; `pdf_to_work.csv` maps all 294 PDFs to 292 works. This changes only the citation graph representation, not the embedding selection or the PDF files.
- For P301, the [publisher's 2014 issue](https://academic.oup.com/rfs/issue/27/7) identifies the appropriate journal record. The local PDF is actually NBER working paper 19612 (November 2013), despite its 2014 filename. [NBER explicitly links it to the 2014 article](https://www.nber.org/papers/w19612). We use that publication as an allowed version proxy; its references may differ. The duplicated 2012 OpenAlex ID is retained as a target alias for the same work.
- PDF bibliographies and footnotes supply **65 manually checked internal citation links** for the eight empty API lists. Each link has its source PDF hash, page, excerpt and review reason. A false substring match and a publication-version ambiguity were rejected. External reference lists for these papers are still incomplete.

Decisions are saved in `configs/citation_graph_manual_review.json`; page evidence, candidate matches and rejected decisions are in `manual_review/`. `src/batill/citation_curation.py` applies them offline and checks selection identity and evidence checksums. The optional extraction switch below reproduces the page evidence and candidate discovery; it does **not** approve candidates automatically.

For direct citation clustering use `reviewed/edges.csv` or `reviewed/citation_graph.graphml`. For comparisons with embeddings join via `reviewed/pdf_to_work.csv`; do not count duplicated PDFs as independent works. For shared-reference clustering, account explicitly for the eight incomplete external bibliographies. Empty or incomplete provider lists should never be interpreted as complete absence of citations.

In [7]:
from batill.citation_curation import export_reviewed_graph, extract_pdf_review_candidates

REEXTRACT_REVIEW_CANDIDATES = False
if REEXTRACT_REVIEW_CANDIDATES:
    candidates = extract_pdf_review_candidates(ROOT, RUN_DIR)
    display(candidates.head())

reviewed_nodes, reviewed_edges, reviewed_graph, reviewed_summary, pdf_to_work = export_reviewed_graph(
    ROOT, RUN_DIR, selection, client,
)
REVIEWED_DIR = RUN_DIR / 'reviewed'
display(pd.Series({k: v for k, v in reviewed_summary.items()
                   if k not in ['weak_component_sizes', 'isolated_pdfs']}).to_frame('value'))
display(pdf_to_work.loc[pdf_to_work.is_duplicate_pdf,
                       ['paper_id', 'canonical_paper_id', 'Key', 'canonical_key']])
display(reviewed_nodes.loc[reviewed_nodes.manual_internal_citations.gt(0),
    ['paper_id', 'query_title', 'manual_internal_citations', 'reference_coverage_note']])
print('Final reviewed graph:', REVIEWED_DIR)
print('Weak component sizes:', reviewed_summary['weak_component_sizes'])

,value
provider,OpenAlex
selected_pdfs,294
approved_work_identities,292
unapproved_pdfs,0
approved_nonempty_reference_lists,284
approved_empty_reference_lists,8
approved_missing_reference_lists,0
internal_citation_edges,2539
all_reference_links,18238
edges_using_version_alias,120


,paper_id,canonical_paper_id,Key,canonical_key
147,P155,P154,4ae6b2cd03aa509c2f37268691984c436f744fdc1802eb...,8f766757876d335aff1db92c0777a7a0cc31b53bc645fd...
284,P308,P020,5fa7d791b3295880209aded5578abd4ffd2102a276778c...,741967ca9a4eb2faed548028285e9e254e8a77147037db...


,paper_id,query_title,manual_internal_citations,reference_coverage_note
31,P032,An Empirical Guide to Investor-Level Private E...,1,partial_internal_pdf_recovery; external_biblio...
62,P064,What Do Different Commercial Data Sets Tell Us...,5,partial_internal_pdf_recovery; external_biblio...
65,P067,Private equity portfolio companies A first loo...,11,partial_internal_pdf_recovery; external_biblio...
71,P073,The Structure of Leveraged Buyouts and the Fre...,9,partial_internal_pdf_recovery; external_biblio...
161,P170,"Deconstructing equity: Public ownership, agenc...",1,partial_internal_pdf_recovery; external_biblio...
173,P187,Evaluating Private Equity performance using St...,9,partial_internal_pdf_recovery; external_biblio...
219,P236,Private equity for pension plans? Evaluating p...,21,partial_internal_pdf_recovery; external_biblio...
244,P263,Does private equity create wealth? the effects...,8,partial_internal_pdf_recovery; external_biblio...


Final reviewed graph: /Users/fdedek/projects/batill/outputs/citation_graph_openalex/9ad47e071ee12c2f/reviewed
Weak component sizes: [286, 3, 1, 1, 1]


In [8]:
manual_review = read_json(ROOT / 'configs/citation_graph_manual_review.json')
source_lines = '\n'.join(f"- {r['paper_id']}: [{r['finding']}]({r['url']})"
                         for r in manual_review['external_identity_sources'])
curated_report = f"""# Manually reviewed citation graph

## Results

- **{reviewed_summary['selected_pdfs']} selected PDFs → {reviewed_summary['graph_work_nodes']} unique works**.
- **{reviewed_summary['unapproved_pdfs']} unresolved PDF-to-work identities** after the recorded review.
- **{reviewed_summary['internal_citation_edges']} directed internal citation links**.
- **{reviewed_summary['manually_added_internal_edges']} links recovered from local PDF references/footnotes**, affecting {reviewed_summary['partially_recovered_reference_lists']} source papers.
- Largest weakly connected component: **{max(reviewed_summary['weak_component_sizes'])} works**; isolated works: **{reviewed_summary['isolated_works']}**.

The original baseline had 2,396 links and held five PDFs. Identity resolution restores links involving three works; the two duplicate PDF pairs each become one node. Manual citation additions are labelled separately.

## Identity decisions

P308 maps to P020: same authors, title and DOI, confirmed by [Wiley](https://onlinelibrary.wiley.com/doi/10.1111/j.1540-6261.2009.01473.x).

P155 maps to P154: same first-page DOI and authors, confirmed by [Wiley](https://onlinelibrary.wiley.com/doi/10.1111/fima.12264). The article appeared online in 2019 and in a 2020 journal issue; this date difference does not indicate different works.

P301 uses OpenAlex W3123679298 (2014), matching the [Oxford journal issue](https://academic.oup.com/rfs/issue/27/7), with W2891860082 as an alternative target identifier. The local PDF is the November 2013 NBER working paper 19612, despite its 2014 filename. [NBER explicitly identifies the 2014 article as its published version](https://www.nber.org/papers/w19612). Its source references come from the 2014 record only, used as a same-work version proxy under the project policy; they may differ from the local PDF.

Additional identity checks:
{source_lines}

## Citation evidence and limits

The eight empty OpenAlex lists are supplemented with 65 manually verified **internal** citations. Each approved link is in manual_citation_evidence.csv with PDF hash, page, title match and surrounding text. The source PDF pages and candidate/rejection audit remain in ../manual_review/. A substring match to P044 was rejected because the cited authors identify P093. A P170→P225 proposal was held because the cited 1993 publication was not verified equivalent to the selected 1991 article.

These are partial reference lists, particularly for external works. No assertion is made that every citation in these PDFs was recovered. For direct citation clustering the graph is usable; for bibliographic coupling these eight incomplete reference profiles require coverage-aware interpretation or a sensitivity analysis. Automatic API identities elsewhere in the corpus were not all manually re-reviewed. Missing edges remain possible throughout the provider data.

## Reproduction

Run section 7 of Citation_graph_OpenAlex.ipynb after its earlier cells. The offline code is src/batill/citation_curation.py; decisions are in configs/citation_graph_manual_review.json. Original API outputs remain in the parent folder. Use this folder's nodes.csv, edges.csv, citation_graph.graphml and adjacency.npz with adjacency_order.csv. pdf_to_work.csv joins every original PDF hash to its canonical graph node. Citation direction is row/source → column/target. Files and review checksums are recorded in run_manifest.json. PDFs and embedding membership remain unchanged.
"""
(REVIEWED_DIR / 'report.md').write_text(curated_report, encoding='utf-8')
display(Markdown(curated_report))

# Manually reviewed citation graph

## Results

- **294 selected PDFs → 292 unique works**.
- **0 unresolved PDF-to-work identities** after the recorded review.
- **2539 directed internal citation links**.
- **65 links recovered from local PDF references/footnotes**, affecting 8 source papers.
- Largest weakly connected component: **286 works**; isolated works: **3**.

The original baseline had 2,396 links and held five PDFs. Identity resolution restores links involving three works; the two duplicate PDF pairs each become one node. Manual citation additions are labelled separately.

## Identity decisions

P308 maps to P020: same authors, title and DOI, confirmed by [Wiley](https://onlinelibrary.wiley.com/doi/10.1111/j.1540-6261.2009.01473.x).

P155 maps to P154: same first-page DOI and authors, confirmed by [Wiley](https://onlinelibrary.wiley.com/doi/10.1111/fima.12264). The article appeared online in 2019 and in a 2020 journal issue; this date difference does not indicate different works.

P301 uses OpenAlex W3123679298 (2014), matching the [Oxford journal issue](https://academic.oup.com/rfs/issue/27/7), with W2891860082 as an alternative target identifier. The local PDF is the November 2013 NBER working paper 19612, despite its 2014 filename. [NBER explicitly identifies the 2014 article as its published version](https://www.nber.org/papers/w19612). Its source references come from the 2014 record only, used as a same-work version proxy under the project policy; they may differ from the local PDF.

Additional identity checks:
- P032: [Title, four authors and 2020 date agree.](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=3764895)
- P067: [Six authors, title and SSRN DOI agree.](https://scholars.duke.edu/publication/1915071)
- P170: [Gilson/Whitehead 2008 Columbia Law Review article confirmed.](https://scholarship.law.columbia.edu/faculty_scholarship/174/)
- P187: [Gredil/Sorensen/Waller and paper title agree.](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=3506847)
- P236: [Published title agrees with PDF; local references used.](https://www.sciencedirect.com/science/article/abs/pii/S0304405X26001078)
- P263: [Publisher hosts Masulis/Thomas article; local PDF footnote citations used.](https://lawreview.uchicago.edu/sites/default/files/76_1_Masulis_Thomas.pdf)

## Citation evidence and limits

The eight empty OpenAlex lists are supplemented with 65 manually verified **internal** citations. Each approved link is in manual_citation_evidence.csv with PDF hash, page, title match and surrounding text. The source PDF pages and candidate/rejection audit remain in ../manual_review/. A substring match to P044 was rejected because the cited authors identify P093. A P170→P225 proposal was held because the cited 1993 publication was not verified equivalent to the selected 1991 article.

These are partial reference lists, particularly for external works. No assertion is made that every citation in these PDFs was recovered. For direct citation clustering the graph is usable; for bibliographic coupling these eight incomplete reference profiles require coverage-aware interpretation or a sensitivity analysis. Automatic API identities elsewhere in the corpus were not all manually re-reviewed. Missing edges remain possible throughout the provider data.

## Reproduction

Run section 7 of Citation_graph_OpenAlex.ipynb after its earlier cells. The offline code is src/batill/citation_curation.py; decisions are in configs/citation_graph_manual_review.json. Original API outputs remain in the parent folder. Use this folder's nodes.csv, edges.csv, citation_graph.graphml and adjacency.npz with adjacency_order.csv. pdf_to_work.csv joins every original PDF hash to its canonical graph node. Citation direction is row/source → column/target. Files and review checksums are recorded in run_manifest.json. PDFs and embedding membership remain unchanged.
